##Birinci DETR modeli

In [ ]:
pip install transformers datasets torchvision albumentations

In [ ]:
!pip install roboflow


In [ ]:

from roboflow import Roboflow
rf = Roboflow(api_key="wHSfaeik5dE3zVrmhk8g")
project = rf.workspace("varroa-j8231").project("varroa-bxxhd")
version = project.version(1)
dataset = version.download("coco")


In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="wHSfaeik5dE3zVrmhk8g")
project = rf.workspace("honeybee").project("honeybee_varroamite")
version = project.version(3)
dataset = version.download("coco")


In [ ]:
from transformers import DetrImageProcessor

processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.


preprocessor_config.json:   0%|          | 0.00/290 [00:00<?, ?B/s]

In [ ]:
# Eski hücreyi bununla değiştirin:
from transformers import DeformableDetrImageProcessor

processor = DeformableDetrImageProcessor.from_pretrained("SenseTime/deformable-detr")

In [ ]:
from transformers import DetrForObjectDetection

In [ ]:
model = DetrForObjectDetection.from_pretrained(
"/content/drive/MyDrive/DETR_Model_1_Veriseti"
)

In [ ]:
from transformers import DeformableDetrForObjectDetection

In [ ]:
model = DeformableDetrForObjectDetection.from_pretrained(
    "/content/drive/MyDrive/Deformable_detr_2_model_birinci_veriseti",
)

In [ ]:
model = DeformableDetrForObjectDetection.from_pretrained(
    "/content/drive/MyDrive/deformable_detr_62_epoch_model",
)

Loading weights:   0%|          | 0/585 [00:00<?, ?it/s]

In [ ]:
import os
import json
from datasets import Dataset, DatasetDict, Image

def load_coco_detection_dataset(image_dir, annotation_file):
    with open(annotation_file, 'r') as f:
        coco_data = json.load(f)

    id_to_image = {img['id']: img for img in coco_data['images']}
    id_to_anns = {}

    for ann in coco_data['annotations']:
        img_id = ann['image_id']
        if img_id not in id_to_anns:
            id_to_anns[img_id] = []

        # Deformable DETR için DÜZELTME:
        # Verideki 1 (Bee) ve 2 (Varroa) değerlerini
        # Modelin beklediği 0 ve 1 formatına çekiyoruz.
        ann_copy = ann.copy()
        ann_copy['category_id'] = ann['category_id'] - 1
        id_to_anns[img_id].append(ann_copy)

    data_list = []
    for img_id, img_info in id_to_image.items():
        img_path = os.path.join(image_dir, img_info['file_name'])
        if os.path.exists(img_path):
            data_list.append({
                "image": img_path,
                "label": {
                    "image_id": img_id,
                    "annotations": id_to_anns.get(img_id, [])
                }
            })

    return Dataset.from_list(data_list).cast_column("image", Image())

# Veriyi yeni ID haritası ile tekrar yüklüyoruz
train_ds = load_coco_detection_dataset("/content/HoneyBee_VarroaMite-3/train", "/content/HoneyBee_VarroaMite-3/train/_annotations.coco.json")
valid_ds = load_coco_detection_dataset("/content/HoneyBee_VarroaMite-3/valid", "/content/HoneyBee_VarroaMite-3/valid/_annotations.coco.json")
dataset = DatasetDict({"train": train_ds, "validation": valid_ds})

In [ ]:
def transform(examples):
    images = [img.convert("RGB") for img in examples["image"]]

    # Veri yükleme fonksiyonunuzdaki 'label' yapısını DETR'ın istediği COCO formatına çekiyoruz
    targets = []
    for i in range(len(images)):
        # load_coco_detection_dataset kullandığınız için 'label' bir sözlük: {'image_id': ..., 'annotations': [...]}
        ann = examples["label"][i]["annotations"]
        img_id = examples["label"][i]["image_id"]
        targets.append({"image_id": img_id, "annotations": ann})

    # return_tensors="pt" diyerek doğrudan PyTorch tensorleri alıyoruz
    encoding = processor(images=images, annotations=targets, return_tensors="pt")

    # ÖNEMLİ: İşlemci çıktısı olan 'pixel_values' ve 'labels'ı doğrudan sözlük olarak döndürüyoruz
    return {
        "pixel_values": encoding["pixel_values"],
        "pixel_mask": encoding["pixel_mask"],
        "labels": encoding["labels"]
    }

# Transform'u uygula
dataset["train"].set_transform(transform)
dataset["validation"].set_transform(transform)

In [ ]:
import torch

def collate_fn(batch):
    # 'pixel_values' listesini tensor'e çevirip istifliyoruz (stack)
    pixel_values = torch.stack([torch.as_tensor(item["pixel_values"]) for item in batch])

    # 'pixel_mask' listesini tensor'e çevirip istifliyoruz
    pixel_mask = torch.stack([torch.as_tensor(item["pixel_mask"]) for item in batch])

    # 'labels' bir liste olmalı, ancak içindeki her bir değer tensor olmalı
    labels = []
    for item in batch:
        new_label = {}
        for k, v in item["labels"].items():
            new_label[k] = torch.as_tensor(v)
        labels.append(new_label)

    return {
        "pixel_values": pixel_values,
        "pixel_mask": pixel_mask,
        "labels": labels
    }

In [ ]:
dataset["validation"].reset_format()

In [ ]:
pip install torchmetrics

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from tqdm import tqdm

# 1. Batch'leri hazırlamak için özel fonksiyon
def collate_fn(batch):
    images = [item["image"].convert("RGB") for item in batch]
    labels = [item["label"] for item in batch]
    return images, labels

# 2. DataLoader kurulumu (Hızın anahtarı burası)
# batch_size: Belleğine göre 8, 16 veya 32 yapabilirsin.
# num_workers: İşlemcinin kaç çekirdeğinin veri hazırlayacağını belirler.
batch_size = 8
test_loader = DataLoader(
    dataset["validation"],
    batch_size=batch_size,
    collate_fn=collate_fn,
    shuffle=False
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()
metric = MeanAveragePrecision(box_format='xyxy', class_metrics=True)


print(f"Test seti {batch_size}'li gruplar halinde değerlendiriliyor...")

for images, labels in tqdm(test_loader):
    # Batch olarak ön işleme ve GPU'ya taşıma
    inputs = processor(images=images, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)

    # Orijinal resim boyutlarını topla (post-process için gerekli)
    target_sizes = torch.tensor([img.size[::-1] for img in images])

    # Model çıktılarını işle (tüm batch için)
    batch_preds = processor.post_process_object_detection(outputs, target_sizes=target_sizes, threshold=0.0)

    # Metrik için Prediction ve Target listelerini hazırla
    formatted_preds = []
    formatted_targets = []

    for i in range(len(images)):
        # Gerçek Etiketler (Target) hazırlama
        target_boxes = []
        target_labels = []
        for ann in labels[i]["annotations"]:
            x, y, w, h = ann["bbox"]
            target_boxes.append([x, y, x + w, y + h])
            target_labels.append(ann["category_id"])

        if not target_boxes: # Eğer resimde nesne yoksa atla (isteğe bağlı)
            continue

        formatted_targets.append({
            "boxes": torch.tensor(target_boxes),
            "labels": torch.tensor(target_labels)
        })

        # Tahminler (Prediction) hazırlama
        formatted_preds.append({
            "boxes": batch_preds[i]["boxes"].cpu(),
            "scores": batch_preds[i]["scores"].cpu(),
            "labels": batch_preds[i]["labels"].cpu()
        })

    # Metriği batch olarak güncelle
    if formatted_preds:
        metric.update(formatted_preds, formatted_targets)

# Sonuçları hesapla ve bas
print("\nMetrikler hesaplanıyor, lütfen bekleyin...")
mAP_results = metric.compute()

print("-" * 30)
print(f"Genel mAP (50-95): {mAP_results['map'].item():.4f}")
print(f"mAP @0.50 (Başarı Skoru): {mAP_results['map_50'].item():.4f}")
print("-" * 30)

Test seti 8'li gruplar halinde değerlendiriliyor...


100%|██████████| 198/198 [01:53<00:00,  1.74it/s]



Metrikler hesaplanıyor, lütfen bekleyin...
------------------------------
Genel mAP (50-95): 0.1193
mAP @0.50 (Başarı Skoru): 0.3905
------------------------------


In [ ]:
print("--- Model Beklentisi ---")
print(f"Model id2label: {model.config.id2label}")

# Veri setindeki orijinal kategorileri json dosyasından kontrol edelim
with open('/content/HoneyBee_VarroaMite-3/valid/_annotations.coco.json', 'r') as f:
    coco_data = json.load(f)

print("\n--- Veri Seti (COCO) Kategorileri ---")
for cat in coco_data['categories']:
    print(f"ID: {cat['id']}, İsim: {cat['name']}")

# Mevcut dataset objesindeki ilk birkaç etiketi görelim
print("\n--- İşlenmiş Dataset Örnekleri ---")
for i in range(3):
    labels = dataset['validation'][i]['label']['annotations']
    ids = [ann['category_id'] for ann in labels]
    print(f"Resim {i} Kategori ID'leri: {ids}")

--- Model Beklentisi ---
Model id2label: {0: 'varroa'}

--- Veri Seti (COCO) Kategorileri ---
ID: 0, İsim: mites
ID: 1, İsim: varroa_mite

--- İşlenmiş Dataset Örnekleri ---
Resim 0 Kategori ID'leri: []
Resim 1 Kategori ID'leri: []
Resim 2 Kategori ID'leri: [0]


In [ ]:
# Sonuçları sözlükten çekelim
precision = mAP_results['map_50'].item() # IoU=0.50'deki Hassasiyet (yaklaşık)
recall = mAP_results['mar_100'].item()    # Ortalama Duyarlılık (100 nesne üzerinden)

# F1 Score Hesaplama (Harmonik Ortalama)
if (precision + recall) > 0:
    f1_score = 2 * (precision * recall) / (precision + recall)
else:
    f1_score = 0

print("\n" + "="*30)
print(f"Hassasiyet (Precision): {precision:.4f}")
print(f"Duyarlılık (Recall):    {recall:.4f}")
print(f"F1 Skoru:              {f1_score:.4f}")
print("="*30)


Hassasiyet (Precision): 0.3905
Duyarlılık (Recall):    0.3538
F1 Skoru:              0.3713


In [ ]:
# Arı (ID 0) içeren bir resmi bulup test edelim
ari_index = -1
for idx, item in enumerate(dataset['validation']):
    if any(ann['category_id'] == 0 for ann in item['label']['annotations']):
        ari_index = idx
        break

if ari_index != -1:
    test_img = dataset['validation'][ari_index]['image']
    test_label = dataset['validation'][ari_index]['label']
    inputs = processor(images=test_img, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    target_size = torch.tensor([test_img.size[::-1]])
    pred = processor.post_process_object_detection(outputs, target_sizes=target_size, threshold=0.0)[0]

    gt_boxes = [[ann['bbox'][0], ann['bbox'][1], ann['bbox'][0]+ann['bbox'][2], ann['bbox'][1]+ann['bbox'][3]] for ann in test_label['annotations']]

    print(f"Resim {ari_index} test ediliyor (Arı içeriyor)")
    visualize_prediction(test_img, {'boxes': torch.tensor(gt_boxes)}, pred, threshold=0.3)
else:
    print("Validasyon setinde ID 0 (Arı) içeren resim bulunamadı!")

Validasyon setinde ID 0 (Arı) içeren resim bulunamadı!


In [ ]:
def calculate_box_stats(dataset_split):
    stats = {0: [], 1: []}
    for item in dataset_split:
        for ann in item['label']['annotations']:
            cat_id = ann['category_id']
            w, h = ann['bbox'][2], ann['bbox'][3]
            area = w * h
            if cat_id in stats:
                stats[cat_id].append(area)

    for cat_id, areas in stats.items():
        if areas:
            avg_area = sum(areas) / len(areas)
            print(f"Sınıf {cat_id} için Ortalama Kutu Alanı: {avg_area:.2f} piksel kare (Toplam {len(areas)} kutu)")
        else:
            print(f"Sınıf {cat_id} için hiç kutu bulunamadı.")

print("Validasyon Seti Kutu İstatistikleri:")
calculate_box_stats(dataset['validation'])

Validasyon Seti Kutu İstatistikleri:
Sınıf 0 için hiç kutu bulunamadı.
Sınıf 1 için Ortalama Kutu Alanı: 343865.53 piksel kare (Toplam 1876 kutu)


In [ ]:
# Modelin hangi ID'ye hangi ismi verdiğini kontrol edelim
print("Model ID-Sınıf Eşleşmesi:")
print(model.config.id2label)

# Veri setindeki ilk birkaç etiketi kontrol edelim
print("\nVeri Setindeki Örnek Etiketler (İşlenmiş):")
for i in range(3):
    example_anns = dataset['validation'][i]['label']['annotations']
    ids = [ann['category_id'] for ann in example_anns]
    print(f"Resim {i} Kategori IDleri: {ids}")

Model ID-Sınıf Eşleşmesi:
{0: 'bee', 1: 'varroa'}

Veri Setindeki Örnek Etiketler (İşlenmiş):
Resim 0 Kategori IDleri: [1, 2, 2]
Resim 1 Kategori IDleri: [1]
Resim 2 Kategori IDleri: [1]


In [ ]:
# En yüksek skorlu tahminleri inceleyelim
top_scores, top_indices = torch.topk(pred['scores'], k=5)
print("En yüksek 5 Güven Skoru:", top_scores.tolist())
print("Bu skorlara ait Sınıf ID'leri:", pred['labels'][top_indices].tolist())

# Eğer skorlar 0.5'in altındaysa model nesneyi tam tanıyamıyor demektir.
if top_scores[0] < 0.5:
    print("\nUyarı: Modelin en yüksek güveni bile %50'nin altında. Model yeterince eğitilmemiş görünüyor.")

En yüksek 5 Güven Skoru: [0.9782513380050659, 0.739309549331665, 0.48210766911506653, 0.001252691145054996, 0.0012521905591711402]
Bu skorlara ait Sınıf ID'leri: [1, 1, 1, 1, 1]


In [ ]:
print("--- Deformable DETR Konfigürasyonu ---")
print(f"Model id2label: {model.config.id2label}")

# Veri setindeki benzersiz kategori ID'lerini bulalım
unique_ids = set()
for item in dataset['validation']:
    for ann in item['label']['annotations']:
        unique_ids.add(ann['category_id'])

print(f"\n--- Veri Seti Analizi ---")
print(f"Veri setinde bulunan kategori ID'leri: {list(unique_ids)}")

if 2 in unique_ids and 2 not in model.config.id2label:
    print("\nHATA TESPİT EDİLDİ: Veri setinde ID: 2 var ama model sadece 0 ve 1 biliyor.")
    print("load_coco_detection_dataset fonksiyonunda 'ann['category_id'] - 1' işlemini geri getirmelisin.")

--- Deformable DETR Konfigürasyonu ---
Model id2label: {0: 'bee', 1: 'varroa'}

--- Veri Seti Analizi ---
Veri setinde bulunan kategori ID'leri: [1, 2]

HATA TESPİT EDİLDİ: Veri setinde ID: 2 var ama model sadece 0 ve 1 biliyor.
load_coco_detection_dataset fonksiyonunda 'ann['category_id'] - 1' işlemini geri getirmelisin.
